# 00 · How it works: the bounds matrix

rxembed never says put atom i at (x, y, z). It states relationships, and they all end up in one place:
RDKit's bounds matrix, an N x N table of (min, max) distances.

That table is internal coordinates. No origin, no orientation. Distance geometry finds 3D coordinates
consistent with it.

Plain RDKit until §4.


In [ ]:
import math

import numpy as np
from rdkit import Chem, DistanceGeometry, rdBase
from rdkit.Chem import rdDistGeom
from rdkit.Geometry import Point3D

np.set_printoptions(precision=2, suppress=True)


def dist(conf, i, j):
    p, q = conf.GetAtomPosition(i), conf.GetAtomPosition(j)
    return math.dist((p.x, p.y, p.z), (q.x, q.y, q.z))


def angle(conf, i, j, k):
    pos = conf.GetPositions()
    a, b = pos[i] - pos[j], pos[k] - pos[j]
    return math.degrees(math.acos(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))))


def smooth(bm, max_tol=0.4):
    "Triangle-smooth in place, growing the tolerance until the matrix is metric-consistent."
    back, tol = bm.copy(), 0.0
    while not DistanceGeometry.DoTriangleSmoothing(bm, tol):
        tol = 1.2 * tol + 0.02
        if tol > max_tol:
            return None
        bm[:] = back
    return tol


def etkdg(seed=0xF00D):
    p = rdDistGeom.ETKDGv3()
    p.randomSeed = seed  # RDKit's own default is -1: the GLOBAL RNG, so results depend on prior consumption
    p.pruneRmsThresh = -1.0
    return p

## 1 · The table

`GetMoleculeBoundsMatrix` gives RDKit's knowledge-derived bounds for every atom pair. Upper triangle
`bm[i][j]` (i<j) is the maximum, lower triangle `bm[j][i]` the minimum. A bond has a tight window, two ends
of a floppy chain a wide one.


In [ ]:
mol = Chem.AddHs(Chem.MolFromSmiles("OCCN"))  # heavy atoms: O0 C1 C2 N3
bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)

print("bounds among the four heavy atoms (upper = max, lower = min):")
print(bm[:4, :4])
print(f"\nO0-C1  bond : {bm[1][0]:.2f} .. {bm[0][1]:.2f} A   (tight - RDKit knows this bond)")
print(f"O0..N3 1-4  : {bm[3][0]:.2f} .. {bm[0][3]:.2f} A   (wide  - a rotatable chain)")

## 2 · A constraint is a window, then a re-smooth

Write a `(min, max)` into the table. Triangle smoothing propagates that edit to every other pair,
so simultaneous constraints share one metric.

Two things below: a tighter window is obeyed more closely, and the realised distance lands near the window
rather than strictly inside it. The matrix biases the embed. §5 is what tightens it.


In [ ]:
def embed_with_window(mol, i, j, lo, hi, n=10, seed=0xF00D):
    "Edit one (min, max) window into the bounds matrix and embed n conformers against it."
    mol = Chem.Mol(mol)
    bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)
    a, b = (i, j) if i < j else (j, i)
    bm[a][b], bm[b][a] = hi, lo  # upper = max, lower = min
    if smooth(bm) is None:
        return mol, [], "infeasible: triangle smoothing could not repair it"
    p = etkdg(seed)
    p.SetBoundsMat(bm)
    return mol, list(rdDistGeom.EmbedMultipleConfs(mol, n, p)), ""


for lo, hi in [(2.6, 3.0), (2.9, 4.5), (1.0, 1.1)]:
    m, ids, err = embed_with_window(mol, 0, 3, lo, hi)
    if err:
        print(f"window ({lo}, {hi}): {err}")
        continue
    d = [dist(m.GetConformer(c), 0, 3) for c in ids]
    print(f"window ({lo}, {hi}): {len(ids):2d} conformers, realised O0..N3 = {min(d):.2f} .. {max(d):.2f} A")

## 3 · Angles are distances

There is no slot for an angle, but an angle i-j-k is fixed once the triangle's three sides are:

$$d_{ik} = \sqrt{d_{ij}^2 + d_{jk}^2 - 2\,d_{ij}\,d_{jk}\cos\theta}$$

`cos` is monotonic on [0°, 180°], so an angle window becomes a distance window: another row. The legs come
from an explicit constraint on that leg if you gave one, else the midpoint of its current bounds.


In [ ]:
def one_three(dij, djk, theta_deg):
    return math.sqrt(dij**2 + djk**2 - 2 * dij * djk * math.cos(math.radians(theta_deg)))


print("angle O0-C1-C2  ->  the O0..C2 distance, for legs 1.43 and 1.52 A:")
for th in (90, 100, 109, 120, 180):
    print(f"   {th:3d} deg  ->  {one_three(1.43, 1.52, th):.3f} A")

bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)
dij = 0.5 * (bm[0][1] + bm[1][0])  # the O0-C1 leg: the midpoint of its bounds IS the bond length
djk = 0.5 * (bm[1][2] + bm[2][1])
lo, hi = one_three(dij, djk, 106), one_three(dij, djk, 112)
bm[0][2], bm[2][0] = hi, lo  # the 1-3 pair, written like any other window
smooth(bm)

p = etkdg()
p.SetBoundsMat(bm)
m = Chem.Mol(mol)
ids = list(rdDistGeom.EmbedMultipleConfs(m, 12, p))
got = [angle(m.GetConformer(c), 0, 1, 2) for c in ids]
print(f"\nlegs {dij:.3f} / {djk:.3f}  ->  1-3 window ({lo:.3f}, {hi:.3f}) A")
print(f"asked 106-112 deg  ->  realised {min(got):.1f} .. {max(got):.1f} deg")

## 4 · Why not `coordMap`?

RDKit's other option pins chosen atoms to absolute Cartesian points. Where it stops:

- points only, so there is no way to state a window (an NCI contact at 2.6-3.0 Å) or an angle
- exact pins can leave triangle smoothing no slack

The cross-fragment complaint is really RDKit's `embedFragmentsSeparately=True` default; flip it and pins
work across fragments.


In [ ]:
def embed_coordmap(mol, cmap, n=8, **kw):
    m, p = Chem.Mol(mol), etkdg()
    p.SetCoordMap(cmap)
    for k, v in kw.items():
        setattr(p, k, v)
    with rdBase.BlockLogs():
        ids = list(rdDistGeom.EmbedMultipleConfs(m, n, p))
    return m, ids


m, ids = embed_coordmap(mol, {0: Point3D(0.0, 0.0, 0.0), 3: Point3D(2.8, 0.0, 0.0)})
d = [dist(m.GetConformer(c), 0, 3) for c in ids]
print(f"(a) one fragment, two atoms pinned : {len(ids)} conformers, O0..N3 = {min(d):.2f}..{max(d):.2f} A (tight)")

two = Chem.AddHs(Chem.MolFromSmiles("CC(=O)O.CCN"))  # heavy: 0-3 | 4-6
cmap = {1: Point3D(0.0, 0.0, 0.0), 6: Point3D(2.7, 0.0, 0.0)}
for tag, kw in [("default", {}), ("embedFragmentsSeparately=False", {"embedFragmentsSeparately": False})]:
    m, ids = embed_coordmap(two, cmap, **kw)
    d = [dist(m.GetConformer(c), 1, 6) for c in ids] if ids else [float("nan")]
    print(f"(b) two fragments, {tag:31s}: {len(ids)} conformers, C1..N6 = {min(d):.2f}..{max(d):.2f} A (asked 2.70)")

probe = Chem.Mol(two)
rdDistGeom.EmbedMolecule(probe, randomSeed=2)
pos = probe.GetConformer().GetPositions()
every = {i: Point3D(*pos[i]) for i in range(two.GetNumAtoms()) if two.GetAtomWithIdx(i).GetAtomicNum() > 1}
m, ids = embed_coordmap(two, every, n=4, embedFragmentsSeparately=False)
print(f"(c) two fragments, all heavy atoms pinned exactly: {len(ids)} conformers; exact pins leave no slack")

## 5 · This is rxembed

There is no second mechanism. A soft window, a TS angle, a hydrogen-bond grip and an octahedral sphere are
all rows in the same table, so they stack. `Constraints` is the struct every builder fills.

One rule: where a real bond path already connects the end atoms, an angle-derived distance intersects the
matrix rather than overwriting it, so the tighter wins. Where the atoms meet only through a stripped metal
bond there is no such path, and the angle is written outright.


In [ ]:
import rxembed.core as rx

confs = rx.embed(Chem.Mol(mol), constrain={(0, 3): 2.8, (0, 1, 2): 109.0}, n=8).minimize()
print("O0..N3:", {k: round(v, 2) for k, v in confs.measure((0, 3)).items()})
print("O0-C1-C2:", {k: round(v, 1) for k, v in confs.measure((0, 1, 2)).items()})

A metal is the same table again. The coordination sphere is a set of M–donor distance windows plus
donor–M–donor angle windows, and nothing about it is a special code path.

In [ ]:
isos = rx.enumerate_isomers(Chem.MolFromSmiles("N->[Pd+2](<-N)(<-[Cl-])<-[Cl-]"), "square_planar")
iso = isos.select(index=1)
iso.summary()
print()

sphere = iso.cons
for (i, j), (lo, hi) in sorted(sphere.distances.items()):
    print(f"  distance {i}-{j}    {lo:.2f} .. {hi:.2f} A")
for (i, j, k), (lo, hi) in sorted(sphere.angles.items()):
    print(f"  angle    {i}-{j}-{k}  {lo:.0f} .. {hi:.0f} deg")

## 6 · Embed biases, the relax holds, the graft is exact

- embed: the matrix biases the seed, approximate and fast
- relax: restrained UFF pulls the seed into the windows, `Conformers.minimize()`
- graft: distance geometry gives ~0.2-0.4 Å internal RMSD on a rigid core, fine for a molecule and wrong
  for a forming bond, so the exact input coordinates are Kabsch-fitted onto each pose and held at 0.000 Å


In [ ]:
ester = Chem.AddHs(Chem.MolFromSmiles("c1ccccc1C(=O)OCCN"))
rdDistGeom.EmbedMolecule(ester, randomSeed=1)  # stand-in for a real TS geometry
core = list(range(6))  # freeze the benzene ring at its input coordinates
ref = ester.GetConformer().GetPositions()[core]

confs = rx.embed(ester, fix=core, n=4, seed=1).minimize()
out = confs.mol.GetConformer(int(confs.ids[0])).GetPositions()[core]

ref_c, out_c = ref - ref.mean(0), out - out.mean(0)
u, _s, vt = np.linalg.svd(ref_c.T @ out_c)
rot = vt.T @ np.diag([1.0, 1.0, np.sign(np.linalg.det(vt.T @ u.T))]) @ u.T
print(f"frozen core RMSD after embed + relax: {np.sqrt(((ref_c @ rot.T - out_c) ** 2).sum(1).mean()):.4f} A")
print(f"the rest of the molecule is free    : {len(confs)} distinct conformers")

## Next

- [`01_basics`](01_basics.ipynb) embedding, and controlling it
- [`02_constraints`](02_constraints.ipynb) distance, angle and plane windows
- [`04_organic_ts`](04_organic_ts.ipynb) a rigid reacting core
